## GPT prompting: example notebook with all prompts

### requires python >= 3.10

## Working with batch mode:

#### Parts 1 (preparation) and 2 (gpt client) have to be run every time the notebook is opened.

#### Part 3 is required if new batch file has to be created

#### Part 4 can be run in parts

    The user can submit the batch job, close the notebook (if need be) and return later (then parts 1 and 2 have to be run again). If this notebook is duplicated then multiple datasets can be processed in parallel (be mindful of the job_id-s and that correct job files are used).

**Since the prompts work in a chain (previous prompt's output is the next prompt's input) then the batch jobs for other prompts should not be created or submitted before previous prompt data has been retrieved and added to the result file.**

If the user wants, then the prompts can all have the same exact data as input without other prompt results (requires changes in code). In that case, multiple batch jobs can be submitted in parallel. That would increase the cost of prompting as the size of the input would not decrease with each prompt.


Code for batch prompting is from here: https://developers.openai.com/cookbook/examples/batch_processing.

Tutorial: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/batch?tabs=global-batch%2Cstandard-input%2Cpython-key&pivots=ai-foundry-portal

The portal for submitting batch jobs: https://ai.azure.com/

In [2]:
#!pip install --upgrade openai

In [1]:
# for auto-reloading extenrnal modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

#%reload_ext autoreload

In [2]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys
from pathlib import Path
import requests

In [3]:
#sys.path.append("../../../")
sys.path.append("../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

# 1. Preparation

In [5]:
INPUT_FOLDER = "../data/datasets_500_all_results/" # "../data/datasets_500_all/"

OUTPUT_FOLDER = "../data/datasets_500_all_results/" # "../data/datasets_500_all/"

AI_CONF_FILE = "../../../v04_verb-case_pattern/minu_code/azure_gb.ini"

BS = 30


In [6]:
a50df = pd.read_csv(os.path.join(INPUT_FOLDER, "A_n50_500_lihtlauseteks/A_n50_500_batch_tagged_final_lihtlaused.csv"), encoding="utf-8", sep=",") 
a80df = pd.read_csv(os.path.join(INPUT_FOLDER, "A_n80_500_lihtlauseteks/A_n80_500_batch_tagged_final_lihtlaused.csv"), encoding="utf-8", sep=",") 
s50df = pd.read_csv(os.path.join(INPUT_FOLDER, "S_n50_500_lihtlauseteks/S_n50_500_batch_tagged_final_lihtlaused.csv"), encoding="utf-8", sep=",") 
s80df = pd.read_csv(os.path.join(INPUT_FOLDER, "S_n80_500_lihtlauseteks/S_n80_500_batch_tagged_final_lihtlaused.csv"), encoding="utf-8", sep=",") 

a50df["initial_cat"] = "a_n50"
a80df["initial_cat"] = "a_n80"
s50df["initial_cat"] = "s_n50"
s80df["initial_cat"] = "s_n80"

In [7]:
df = pd.concat([a50df, a80df, s50df, s80df], ignore_index=True)

In [9]:
df.to_csv(os.path.join(OUTPUT_FOLDER, "samples_for_simple_sentences.csv"),  encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [29]:


INPUT_FILE = os.path.join(OUTPUT_FOLDER, "samples_for_simple_sentences.csv")
INPUT_FILE_BASE = Path(INPUT_FILE).stem

In [43]:
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "samples_for_simple_sentences_results.csv")

In [65]:
def merge_data(df1_base, df_answ, tag):
    key_cols = ['sentence_id','head_id', 'head_loc', "verb", "verb_compound", "morph_case", "form"]

    df_selected = df_answ[key_cols + [tag]].copy()

    df1_base['verb_compound'] = df1_base['verb_compound'].astype('string').str.strip()
    df_selected['verb_compound'] = df_selected['verb_compound'].astype('string').str.strip()

    # Merge df1 with df2_selected etc
    merged_df = df1_base.merge(df_selected, on=key_cols, how='left')

    #merged_df[tag] = merged_df[tag].fillna("")
    
    return merged_df

# 2. GPT client

## GPT jaoks vajalik

In [27]:
config = configparser.ConfigParser()

status = config.read(AI_CONF_FILE) 
assert status == [AI_CONF_FILE]

#API_VERSION = config['azure-configuration']['api_version']
#AZURE_ENDPOINT = config['azure-configuration']['api_base']
#SUBSCRIPTION_KEY = config['azure-configuration']['api_key'].strip()
#model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

In [21]:
SS_PROMPT = """
You are an expert Estonian linguist.

Your task is to convert Estonian compound and complex sentences into a sequence of simple sentences (lihtlaused).

Rules:

1. Preserve all factual information from the original sentence.
2. Split the sentence into the smallest natural set of simple sentences.
3. Each output sentence must contain only one main proposition.
4. Make implicit information explicit when necessary.
5. Resolve relative clauses ("kes", "mis", "mida", "kus", etc.) into independent simple sentences.
6. Preserve names, dates, numbers, and other details exactly.
7. Do not add information that is not inferable from the original sentence.
8. Use grammatically correct and natural Estonian.
9. Output only the resulting simple sentences.
10. Put each simple sentence on its own line.
11. If the input is already a simple sentence, return it unchanged.
12. Prefer semantic decomposition over syntactic decomposition. If a clause expresses a separate fact, produce a separate simple sentence even if this requires repeating nouns instead of using pronouns.

INPUT
- "few_shots": labeled examples
- "batch": unlabeled examples to classify
Each example contains:
- "id": unique item identifier
- "i": input sentence
- "o": output sentences


OUTPUT FORMAT
Return ONLY a valid JSON object:
{"results":[
{
  "idx": <same idx as input>,
  "o": [
    "<simple sentence 1>",
    "<simple sentence 2>"
  ]
}]
- "results" must be an array
- each output item must contain:
  - "id": copied exactly from the corresponding batch item
  - "o": simple sentences
- the output item with id X must correspond to the batch item with id X
- preserve ids exactly
- do not invent new ids
- do not omit ids
- the number of elements in "results" MUST equal the number of items in "batch"
- the i-th element in "results" corresponds exactly to batch[i]
- no items may be skipped or reordered
- no markdown
- no explanations
- no extra text
"""


FEW_SHOTS = [
            user_message(idx=0, i="Katastroofikohas möllas mitu tundi tulekahju , mida umbes 20-kraadises külmas kustutasid kõik linna tuletõrjekomandod ."),
            assistant_message(idx=0, o= ["Katastroofikohas möllas mitu tundi tulekahju.","Väljas oli umbes 20 kraadi külma.","Kõik linna tuletõrjekomandod kustutasid tulekahju."]),
            
            user_message(idx=1, i="Mees , kes töötas aastaid õpetajana , kolis eelmisel aastal Tartusse ."),
            assistant_message(idx=1, o= ["Mees töötas aastaid õpetajana.", "Mees kolis eelmisel aastal Tartusse."]),
            
            user_message(idx=2, i="Mari ostis poest piima ."),
            assistant_message(idx=2, o= [ "Mari ostis poest piima."]),
            
            user_message(idx=3, i="Kui rong saabus jaama , hakkasid reisijad väljuma ."),
            assistant_message(idx=3, o= ["Rong saabus jaama.", "Reisijad hakkasid väljuma."]),
    
]


# 3. Functions for batch mode

In [23]:
def make_task(idx, my_batch, few_shots, system_prompt, deployment):
    """Make single task"""
    
    structured_few_shots = []

    i = 0
    while i < len(few_shots):
        user_msg = few_shots[i]
        assistant_msg = few_shots[i + 1] if i + 1 < len(few_shots) else None

        if assistant_msg is None:
            break

        structured_few_shots.append({
            "idx": user_msg["content"]["idx"],
            "i": user_msg["content"]["i"],
            "idx": assistant_msg["content"]["idx"],
            "o": assistant_msg["content"].get("o", "")
        })
        i += 2
    
    user_payload = {
            "few_shots": structured_few_shots, #few_shots,
            "batch": my_batch
        }

    task = {
        "custom_id": f"task-{idx}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": deployment,
            "temperature": 0,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": json.dumps(user_payload, ensure_ascii=False)
                }
            ],
        }
    }

    return task



# Creating an array of json tasks
def task_array(df, few_shots, system_prompt, deployment):
    """Make an array of tasks."""
    tasks = []

    task_batch = []

    rows = df.to_dict(orient="records")

    index = 0
    for df_batch in tqdm(chunk_data(rows, size=BS)):

        batch = []
        batch_data = []
        for k, ex in enumerate(df_batch):
            #batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))
            batch.append( {"id": k, "i": ex["sentence"]} )
            batch_data.append((ex["sentence_id"], ex["head_id"], ex["head_loc"],
                              ex["verb"], ex["verb_compound"], ex["morph_case"], 
                              ex["sentence"], ex["form"]))

        task = make_task(index, batch, few_shots, system_prompt, deployment)
        task_batch.append((task["custom_id"], batch_data))

        tasks.append(task)
        index += 1
        
    return tasks, task_batch

# 4. Prompting


## MAKE SIMPLE SENTENCES

In [84]:
# algne andmefail
#df1 = pd.read_csv(INPUT_FILE, encoding="utf-8",  sep=",")
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = [] #"ner_tag,timex_tag".split(",")

df = df1.copy().iloc[30:]
#df = df.sample(frac=1)

prompt_tag = "simple_sentences_gpt4o"

In [69]:
df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,opinion,...,A,E,T,L1,L2,L,S,tag,initial_cat,simple_sentences_gpt4o
30,1482904,2360290,1,soovima,NaN,el,Eesti,Eestist,"Eestist soovivad Venemaale või mõnda teise SRÜ riiki ümber asuda keskmisest haritumad mittekodanikud , selgub eile avalikustatud uuringust .",NaN,...,no,no,no,yes,no,yes,no,L,a_n50,NaN
31,12565181,20108405,15,olema,tarvis,ad,suusataja,suusatajail,Eesti suusaliidu peasekretär Kaarel Zilmer säras kui pühademuna : vaatamata Kristina Šmiguni õnnetusele pole suusatajail tarvis koju tagauksest siseneda .,NaN,...,yes,no,no,no,no,no,no,A,a_n50,NaN
32,3537862,5697343,4,tahtma,NaN,el,kaaslane,kaaslasest,Heast ja kannatlikust kaaslasest ei tahaks lapsed ja kasvatajad aga loobuda .,NaN,...,yes,no,no,no,no,no,no,A,a_n50,NaN
33,2868731,4600234,25,piinama,NaN,el,sügis,sügisest,"Päris sama koosseisuga ei saaks neljapaat kuidagi aga edasi sõita , kuna eessõudja Andrei Jämsä pole lahti saanud kõhu- ja seljavaludest , mis teda sügisest saati piinavad .",NaN,...,no,no,yes,no,no,no,no,T,a_n50,NaN
34,4530501,7282195,2,köitma,NaN,el,liikluskorraldaja,liikluskorraldajatest,"Tallinna liikluskorraldajatest ja parkimisametnikest , maanteeameti liiklusohutusspetsialistidest , liikluspolitseinikest ja inseneribüroo Stratum töötajatest koosneva ümarlaua tähelepanu köitsid viimastel kuudel toimunud järjestikused ränkade tagajärgedega otsasõidud Tallinna vöötradadel .",???,...,yes,no,no,no,no,no,no,A,a_n50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,15931669,24783119,1,põdema,NaN,adit,maksavähk,Maksavähki,Maksavähki põdenud 56-aastane Meyer-Wölden suri esmaspäeval Münchenis .,NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN
796,4651373,7474900,10,surema,NaN,adit,veesurm,veesurma,"Seni oli väikseima uppunute arvuga aasta 1987 , mil veesurma suri 89 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN
797,7489204,12019665,4,surema,NaN,adit,narkosurm,narkosurma,"Kuid Bangs suri narkosurma juba kaks aastat hiljem , kaks aastat enne Madonnat , ning sel moel teame meie nüüd temast rohkem .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN
798,2463299,3948849,11,surema,NaN,adit,kõhulahtisus,kõhulahtisusse,"Rohkem kui 3 miljonit alla 5-aastast last sureb igal aastal kõhulahtisusse , mille taustal võib ainult imestada , millise paanika põhjustas Euroopas hullulehmatõbi , mille tagajärjel suri enam kui 10 aasta jooksul vähem kui 100 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN


In [55]:
# make task array
tasks, task_batch = task_array(df, FEW_SHOTS, SS_PROMPT, DEPLOYMENT)

26it [00:00, 8455.60it/s]


In [56]:
# save task array and extra info to files

file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks.jsonl")

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')
        
f2 = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_data.jsonl")

with open(f2, 'w') as file:
    for obj in task_batch:
        file.write(json.dumps(obj,ensure_ascii=False) + '\n')

#### Retrieving results

In [58]:
result_file_name = os.path.join(OUTPUT_FOLDER, f"{INPUT_FILE_BASE}_{prompt_tag}_batch_tasks_results.jsonl")

In [59]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [60]:
len(results)

26

In [61]:
merged_data = []

for res in results:
    task_id = res['custom_id']
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    answers = json.loads(result)["results"]

    task_data = None
    for row in task_batch:
        if row[0] == task_id:
            task_data = row
            break
    
    if len(answers) == len(task_data[1]):
        c = 0
        for task, answer in zip(task_data[1], answers):
            if c == answer["id"]:
                d = task + (answer["o"],)
                merged_data.append(d)
                c+= 1
            else:
                print(f"missing an answer for task {task_id} id {c}")
                d = task + ("",)
                merged_data.append(d)
    else:
        print(f"{task_id}:", (len(answers)), "vs", (len(task_data[1])))
        
    if task_data is None:
        print("No task_data for task_id", task_id)
        
cols = ["sentence_id","head_id","head_loc", "verb","verb_compound","morph_case","sentence","form", prompt_tag]
mdf = pd.DataFrame(merged_data, columns = cols)

In [76]:
mdf

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,sentence,form,simple_sentences_gpt4o
0,6879142,11062320,15,surema,NaN,adit,"Wimaks kiskusid nemmad temmal kõrri lõua alt wälja , kus ta sure walloga ärra surri .",surri,"[Nemmad kiskusid Wimaks temmal kõrri lõua alt välja., Ta suri walloga.]"
1,192887,318505,14,nakatuma,NaN,adit,Ammu enne hullu gripilaine jõudmist Eestisse on päris paljud eestlased nakatunud teise moodsasse tõppe - Sudokusse .,tõppe,"[Hull gripilaine ei olnud veel Eestisse jõudnud., Paljud eestlased olid nakatunud teise moodsasse tõppe., See tõbi on Sudoku.]"
2,6158711,9885944,26,nakatuma,NaN,adit,"NEW YORK , 30. märts ( AP-EPLO ) - Tervishoiuametnikud on tuvastanud juba mitmeid patsiente , kes on ilmselt nakatunud harvaesinevasse ja ravimitele allumatusse HI-viiruse tüvve , kuid pole selge , kas need juhtumid on seotud .",tüvve,"[NEW YORK, 30. märts (AP-EPLO) - Tervishoiuametnikud on tuvastanud mitmeid patsiente., Need patsiendid on ilmselt nakatunud harvaesinevasse HI-viiruse tüvve., See tüvi ei allu ravimitele., Pole selge, kas need juhtumid on omavahel seotud.]"
3,933744,1487131,11,surema,NaN,adit,84-aastane Bandaranaike osales eile oma kodumaakonnas parlamendivalimistel ja suri tagasiteel pealinna Colombosse .,pealinna,"[84-aastane Bandaranaike osales eile oma kodumaakonnas parlamendivalimistel., Ta suri tagasiteel pealinna Colombosse.]"
4,6215791,9983047,5,surema,NaN,adit,"Kui parafraseerida Monty Pythoni sketši surnud papagoist , siis tuleb Tony Blair Tallinna sõnumiga "" It is not dead , it's merely resting "" - s.t eesistumisel pole häda midagi , tuleb vaid oodata .",sketši,"[Tony Blair tuleb Tallinna sõnumiga., Sõnum on ""It is not dead, it's merely resting""., See tähendab, et eesistumisel pole häda midagi., Tuleb vaid oodata.]"
...,...,...,...,...,...,...,...,...,...
765,1004152,1599249,9,tulenema,NaN,el,"Brutotulu kasv omakorda tuleneb uute kaupluste ja osakonna avamisest käesoleval majandusaastal , millega pole kaasnenud samaväärset tegevuskulude kasvu ning samuti on 3,2 miljoni krooni võrra vähenenud intressikulud .",avamisest,"[Brutotulu kasv tuleneb uute kaupluste avamisest käesoleval majandusaastal., Brutotulu kasv tuleneb uute osakondade avamisest käesoleval majandusaastal., Uute kaupluste ja osakondade avamisega pole kaasnenud samaväärset tegevuskulude kasvu., Intressikulud on vähenenud 3,2 miljoni krooni võrra.]"
766,18091758,27520386,25,johtuma,NaN,el,"Käesolevate suuniste alusel antud abi ei või ühendada muude asutamislepingu artikli 87 lõike 1 tähenduses riigiabi vormidega või muude ühenduse rahastamisvormidega , kui niisugusest kattuvusest johtub suurem toetuse määr , kui käesolevates suunistes ette on nähtud .",kattuvusest,"[Käesolevate suuniste alusel antud abi ei või ühendada muude riigiabi vormidega., Käesolevate suuniste alusel antud abi ei või ühendada muude ühenduse rahastamisvormidega., Sellisest kattuvusest ei tohi johtu suurem toetuse määr, kui käesolevates suunistes ette on nähtud.]"
767,2326922,3726261,1,veenduma,NaN,in,"Iisraelis ollakse aga veendunud , et Araabia Liiga tippkohtumiselt pole mõtet oodatagi rahupakkumisi , kirjutab Jerusalem Post .",Iisraelis,"[Iisraelis ollakse veendunud, et Araabia Liiga tippkohtumiselt pole mõtet oodata rahupakkumisi., Jerusalem Post kirjutab sellest.]"
768,339376,524536,14,sattuma,NaN,adit,"Ka mehe vanaema uskus Jumalasse , kuid see ei kandunud mehe maailma ( jutustusse satub see teave ühe koerustüki kaudu ) .",jutustusse,"[Mehe vanaema uskus Jumalasse., See ei kandunud mehe maailma., See teave satub jutustusse ühe koerustüki kaudu.]"


In [85]:
len(mdf)

770

In [86]:
df1

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,opinion,...,A,E,T,L1,L2,L,S,tag,initial_cat,simple_sentences_gpt4o
0,5399131,8658564,1,katsuma,NaN,all,kõrbetaim,Kõrbetaimedele,"Kõrbetaimedele katsu leida korteri kõige valgem koht , hämarate vihmametsade asukad aga ei pea olema isegi mitte aknalaual .",NaN,...,no,no,no,yes,no,yes,no,L,a_n50,"['Kõrbetaimedele tuleb leida korteri kõige valgem koht.', 'Hämarate vihmametsade asukad ei pea olema isegi mitte aknalaual.']"
1,15270292,23846411,2,õhkuma,NaN,el,PR-pude,PR-pudemetest,"Nendest PR-pudemetest õhkub sellist jõulisust ja suursugusust , et ” Siberi habemeajaja ” muutus sündmuseks veel enne ekraanidele jõudmist .",NaN,...,no,no,no,no,no,no,no,NaN,a_n50,"['Nendest PR-pudemetest õhkub jõulisust.', 'Nendest PR-pudemetest õhkub suursugusust.', 'Film ""Siberi habemeajaja"" muutus sündmuseks veel enne ekraanidele jõudmist.']"
2,15331149,23942314,6,ennustama,NaN,all,netifoorum,netifoorumitele,mingit tehnilist hüpet ma küll netifoorumitele ei ennusta .,NaN,...,no,no,no,no,yes,yes,no,L,a_n50,['Ma ei ennusta netifoorumitele mingit tehnilist hüpet.']
3,12437255,19912298,5,suurenema,NaN,all,palk,palgale,"Sellega suurenes nende tähelepanu palgale , mis töö tegelikku intensiivsust ja tulemuslikkust arvestades pole kuigi palju madalam erasektori keskastme töötaja palgast .",NaN,...,no,no,no,no,no,no,no,NaN,a_n50,"['Nende tähelepanu palgale suurenes.', 'Palk ei ole töö tegelikku intensiivsust ja tulemuslikkust arvestades kuigi palju madalam erasektori keskastme töötaja palgast.']"
4,3753353,6044920,2,pöörduma,NaN,all,lend,lennule,Järgmisele lennule pöördub lennuk Tallinnast Kopenhaageni suunas .,NaN,...,no,yes,no,no,no,no,no,E,a_n50,"['Lennuk pöördub järgmisele lennule.', 'Lennuk suundub Tallinnast Kopenhaagenisse.']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,15931669,24783119,1,põdema,NaN,adit,maksavähk,Maksavähki,Maksavähki põdenud 56-aastane Meyer-Wölden suri esmaspäeval Münchenis .,NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN
796,4651373,7474900,10,surema,NaN,adit,veesurm,veesurma,"Seni oli väikseima uppunute arvuga aasta 1987 , mil veesurma suri 89 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN
797,7489204,12019665,4,surema,NaN,adit,narkosurm,narkosurma,"Kuid Bangs suri narkosurma juba kaks aastat hiljem , kaks aastat enne Madonnat , ning sel moel teame meie nüüd temast rohkem .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN
798,2463299,3948849,11,surema,NaN,adit,kõhulahtisus,kõhulahtisusse,"Rohkem kui 3 miljonit alla 5-aastast last sureb igal aastal kõhulahtisusse , mille taustal võib ainult imestada , millise paanika põhjustas Euroopas hullulehmatõbi , mille tagajärjel suri enam kui 10 aasta jooksul vähem kui 100 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,NaN


In [98]:
assert len(mdf) == len(df)

if len(mdf) == len(df):
    merged_df = merge_data(df1, mdf, prompt_tag)
    merged_df[f"{prompt_tag}_x"] = merged_df[f"{prompt_tag}_x"].fillna(merged_df[f"{prompt_tag}_y"])
    merged_df = merged_df.drop(columns=[f"{prompt_tag}_y"])
    merged_df = merged_df.rename(columns={f"{prompt_tag}_x": prompt_tag})


In [99]:
merged_df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,opinion,...,A,E,T,L1,L2,L,S,tag,initial_cat,simple_sentences_gpt4o
0,5399131,8658564,1,katsuma,<NA>,all,kõrbetaim,Kõrbetaimedele,"Kõrbetaimedele katsu leida korteri kõige valgem koht , hämarate vihmametsade asukad aga ei pea olema isegi mitte aknalaual .",NaN,...,no,no,no,yes,no,yes,no,L,a_n50,"['Kõrbetaimedele tuleb leida korteri kõige valgem koht.', 'Hämarate vihmametsade asukad ei pea olema isegi mitte aknalaual.']"
1,15270292,23846411,2,õhkuma,<NA>,el,PR-pude,PR-pudemetest,"Nendest PR-pudemetest õhkub sellist jõulisust ja suursugusust , et ” Siberi habemeajaja ” muutus sündmuseks veel enne ekraanidele jõudmist .",NaN,...,no,no,no,no,no,no,no,NaN,a_n50,"['Nendest PR-pudemetest õhkub jõulisust.', 'Nendest PR-pudemetest õhkub suursugusust.', 'Film ""Siberi habemeajaja"" muutus sündmuseks veel enne ekraanidele jõudmist.']"
2,15331149,23942314,6,ennustama,<NA>,all,netifoorum,netifoorumitele,mingit tehnilist hüpet ma küll netifoorumitele ei ennusta .,NaN,...,no,no,no,no,yes,yes,no,L,a_n50,['Ma ei ennusta netifoorumitele mingit tehnilist hüpet.']
3,12437255,19912298,5,suurenema,<NA>,all,palk,palgale,"Sellega suurenes nende tähelepanu palgale , mis töö tegelikku intensiivsust ja tulemuslikkust arvestades pole kuigi palju madalam erasektori keskastme töötaja palgast .",NaN,...,no,no,no,no,no,no,no,NaN,a_n50,"['Nende tähelepanu palgale suurenes.', 'Palk ei ole töö tegelikku intensiivsust ja tulemuslikkust arvestades kuigi palju madalam erasektori keskastme töötaja palgast.']"
4,3753353,6044920,2,pöörduma,<NA>,all,lend,lennule,Järgmisele lennule pöördub lennuk Tallinnast Kopenhaageni suunas .,NaN,...,no,yes,no,no,no,no,no,E,a_n50,"['Lennuk pöördub järgmisele lennule.', 'Lennuk suundub Tallinnast Kopenhaagenisse.']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,15931669,24783119,1,põdema,<NA>,adit,maksavähk,Maksavähki,Maksavähki põdenud 56-aastane Meyer-Wölden suri esmaspäeval Münchenis .,NaN,...,no,no,no,no,no,no,yes,S,s_n80,"[56-aastane Meyer-Wölden põdes maksavähki., Ta suri esmaspäeval., Ta suri Münchenis.]"
796,4651373,7474900,10,surema,<NA>,adit,veesurm,veesurma,"Seni oli väikseima uppunute arvuga aasta 1987 , mil veesurma suri 89 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,"[1987. aasta oli seni väikseima uppunute arvuga aasta., Sellel aastal suri veesurma 89 inimest.]"
797,7489204,12019665,4,surema,<NA>,adit,narkosurm,narkosurma,"Kuid Bangs suri narkosurma juba kaks aastat hiljem , kaks aastat enne Madonnat , ning sel moel teame meie nüüd temast rohkem .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,"[Bangs suri narkosurma., Ta suri kaks aastat pärast Madonnat., Me teame nüüd temast rohkem.]"
798,2463299,3948849,11,surema,<NA>,adit,kõhulahtisus,kõhulahtisusse,"Rohkem kui 3 miljonit alla 5-aastast last sureb igal aastal kõhulahtisusse , mille taustal võib ainult imestada , millise paanika põhjustas Euroopas hullulehmatõbi , mille tagajärjel suri enam kui 10 aasta jooksul vähem kui 100 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,"[Rohkem kui 3 miljonit alla 5-aastast last sureb igal aastal kõhulahtisusse., Hullulehmatõbi põhjustas Euroopas paanikat., Hullulehmatõve tagajärjel suri enam kui 10 aasta jooksul vähem kui 100 inimest.]"


In [100]:
merged_df = merged_df[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'opinion',
       'simple_sentences', 'simple_sentences_gpt4o', 'tags', 'timex_tag', 'ekilex_tag', 'ner_tag', 'A',
       'E', 'T', 'L1', 'L2', 'L', 'S', 'tag', 'initial_cat'
       ]]

In [101]:
merged_df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,opinion,...,ner_tag,A,E,T,L1,L2,L,S,tag,initial_cat
0,5399131,8658564,1,katsuma,<NA>,all,kõrbetaim,Kõrbetaimedele,"Kõrbetaimedele katsu leida korteri kõige valgem koht , hämarate vihmametsade asukad aga ei pea olema isegi mitte aknalaual .",NaN,...,NaN,no,no,no,yes,no,yes,no,L,a_n50
1,15270292,23846411,2,õhkuma,<NA>,el,PR-pude,PR-pudemetest,"Nendest PR-pudemetest õhkub sellist jõulisust ja suursugusust , et ” Siberi habemeajaja ” muutus sündmuseks veel enne ekraanidele jõudmist .",NaN,...,NaN,no,no,no,no,no,no,no,NaN,a_n50
2,15331149,23942314,6,ennustama,<NA>,all,netifoorum,netifoorumitele,mingit tehnilist hüpet ma küll netifoorumitele ei ennusta .,NaN,...,NaN,no,no,no,no,yes,yes,no,L,a_n50
3,12437255,19912298,5,suurenema,<NA>,all,palk,palgale,"Sellega suurenes nende tähelepanu palgale , mis töö tegelikku intensiivsust ja tulemuslikkust arvestades pole kuigi palju madalam erasektori keskastme töötaja palgast .",NaN,...,NaN,no,no,no,no,no,no,no,NaN,a_n50
4,3753353,6044920,2,pöörduma,<NA>,all,lend,lennule,Järgmisele lennule pöördub lennuk Tallinnast Kopenhaageni suunas .,NaN,...,NaN,no,yes,no,no,no,no,no,E,a_n50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,15931669,24783119,1,põdema,<NA>,adit,maksavähk,Maksavähki,Maksavähki põdenud 56-aastane Meyer-Wölden suri esmaspäeval Münchenis .,NaN,...,NaN,no,no,no,no,no,no,yes,S,s_n80
796,4651373,7474900,10,surema,<NA>,adit,veesurm,veesurma,"Seni oli väikseima uppunute arvuga aasta 1987 , mil veesurma suri 89 inimest .",NaN,...,NaN,no,no,no,no,no,no,yes,S,s_n80
797,7489204,12019665,4,surema,<NA>,adit,narkosurm,narkosurma,"Kuid Bangs suri narkosurma juba kaks aastat hiljem , kaks aastat enne Madonnat , ning sel moel teame meie nüüd temast rohkem .",NaN,...,NaN,no,no,no,no,no,no,yes,S,s_n80
798,2463299,3948849,11,surema,<NA>,adit,kõhulahtisus,kõhulahtisusse,"Rohkem kui 3 miljonit alla 5-aastast last sureb igal aastal kõhulahtisusse , mille taustal võib ainult imestada , millise paanika põhjustas Euroopas hullulehmatõbi , mille tagajärjel suri enam kui 10 aasta jooksul vähem kui 100 inimest .",NaN,...,NaN,no,no,no,no,no,no,yes,S,s_n80


In [102]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [103]:
len(merged_df[merged_df[prompt_tag]!=""])

800

In [52]:
merged_df

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,opinion,...,A,E,T,L1,L2,L,S,tag,initial_cat,simple_sentences_gpt4o
0,5399131,8658564,1,katsuma,<NA>,all,kõrbetaim,Kõrbetaimedele,"Kõrbetaimedele katsu leida korteri kõige valgem koht , hämarate vihmametsade asukad aga ei pea olema isegi mitte aknalaual .",NaN,...,no,no,no,yes,no,yes,no,L,a_n50,"[Kõrbetaimedele tuleb leida korteri kõige valgem koht., Hämarate vihmametsade asukad ei pea olema isegi mitte aknalaual.]"
1,15270292,23846411,2,õhkuma,<NA>,el,PR-pude,PR-pudemetest,"Nendest PR-pudemetest õhkub sellist jõulisust ja suursugusust , et ” Siberi habemeajaja ” muutus sündmuseks veel enne ekraanidele jõudmist .",NaN,...,no,no,no,no,no,no,no,NaN,a_n50,"[Nendest PR-pudemetest õhkub jõulisust., Nendest PR-pudemetest õhkub suursugusust., Film ""Siberi habemeajaja"" muutus sündmuseks veel enne ekraanidele jõudmist.]"
2,15331149,23942314,6,ennustama,<NA>,all,netifoorum,netifoorumitele,mingit tehnilist hüpet ma küll netifoorumitele ei ennusta .,NaN,...,no,no,no,no,yes,yes,no,L,a_n50,[Ma ei ennusta netifoorumitele mingit tehnilist hüpet.]
3,12437255,19912298,5,suurenema,<NA>,all,palk,palgale,"Sellega suurenes nende tähelepanu palgale , mis töö tegelikku intensiivsust ja tulemuslikkust arvestades pole kuigi palju madalam erasektori keskastme töötaja palgast .",NaN,...,no,no,no,no,no,no,no,NaN,a_n50,"[Nende tähelepanu palgale suurenes., Palk ei ole töö tegelikku intensiivsust ja tulemuslikkust arvestades kuigi palju madalam erasektori keskastme töötaja palgast.]"
4,3753353,6044920,2,pöörduma,<NA>,all,lend,lennule,Järgmisele lennule pöördub lennuk Tallinnast Kopenhaageni suunas .,NaN,...,no,yes,no,no,no,no,no,E,a_n50,"[Lennuk pöördub järgmisele lennule., Lennuk suundub Tallinnast Kopenhaagenisse.]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,15931669,24783119,1,põdema,<NA>,adit,maksavähk,Maksavähki,Maksavähki põdenud 56-aastane Meyer-Wölden suri esmaspäeval Münchenis .,NaN,...,no,no,no,no,no,no,yes,S,s_n80,
796,4651373,7474900,10,surema,<NA>,adit,veesurm,veesurma,"Seni oli väikseima uppunute arvuga aasta 1987 , mil veesurma suri 89 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,
797,7489204,12019665,4,surema,<NA>,adit,narkosurm,narkosurma,"Kuid Bangs suri narkosurma juba kaks aastat hiljem , kaks aastat enne Madonnat , ning sel moel teame meie nüüd temast rohkem .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,
798,2463299,3948849,11,surema,<NA>,adit,kõhulahtisus,kõhulahtisusse,"Rohkem kui 3 miljonit alla 5-aastast last sureb igal aastal kõhulahtisusse , mille taustal võib ainult imestada , millise paanika põhjustas Euroopas hullulehmatõbi , mille tagajärjel suri enam kui 10 aasta jooksul vähem kui 100 inimest .",NaN,...,no,no,no,no,no,no,yes,S,s_n80,


In [32]:
df1[~df1["opinion"].isna()]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,opinion,...,ner_tag,A,E,T,L1,L2,L,S,tag,initial_cat
10,1518677,2416707,2,suutma,NaN,el,koht,kohast,Hoolimata kohast riikliku naftafirma juhatuses ning kuulumisest parlamenti pole presidendi poeg suutnud endale tõsist mainet kujundada .,???,...,NaN,no,no,no,yes,no,yes,no,L,a_n50
13,1708610,2719631,13,leidma,NaN,all,audiitor,audiitorile,Prokuröri hinnangul on Kallase ( 52 ) süü Eesti Panga juhina keskpanga audiitorile valeandmete esitamises kohtus tõendamist leidnud ning seetõttu tuleb teda karistada rahatrahviga .,???,...,NaN,yes,no,no,no,no,no,no,A,a_n50
14,16460744,25407442,2,tahtma,NaN,el,igavus,igavusest,"Tahaks igavusest mingit nende mängu proovida , koodi oleks vaja korgi alt .",???,...,NaN,no,no,no,no,no,no,yes,S,a_n50
18,5269131,8452143,14,lööma,NaN,el,Panionio,Panioniost,"Täisedu , üheksa võiduga jätkab üllataja Ateena Peristeri , kes viimati lõi Ateena Panioniost 78 : 76.",syntax,...,ORG,no,no,no,yes,no,yes,no,L,a_n50
25,3449293,5553715,5,võitma,NaN,all,kaitsetegevus,kaitsetegevusele,Kreeka võitis tänu hingestatud kaitsetegevusele ning Efthimios Rentziase ja Nikos Ekonomou heale mängule .,???,...,NaN,no,no,no,no,no,no,no,NaN,a_n50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
773,15364971,23997877,11,surema,NaN,adit,vang,vangi,"Usbekistanis Andijoni vanglas suri novembris piinamise tagajärjel kaks islamiäärmusluses süüdistatud vangi , ütlesid meeste sugulased ja inimõiguslased .",syntax,...,NaN,yes,no,no,no,no,no,no,A,s_n80
775,5764928,9248055,12,surema,NaN,adit,heinakuhi,heinakuhja,""" Buridanid "" said oma nime eesli järgi , kes kahe heinakuhja vahel nälga suri .",syntax,...,NaN,no,no,no,yes,no,yes,no,L,s_n80
780,6879142,11062320,15,surema,NaN,adit,surr,surri,"Wimaks kiskusid nemmad temmal kõrri lõua alt wälja , kus ta sure walloga ärra surri .",syntax,...,NaN,no,no,no,no,no,no,yes,S,s_n80
783,933744,1487131,11,surema,NaN,adit,pealinn,pealinna,84-aastane Bandaranaike osales eile oma kodumaakonnas parlamendivalimistel ja suri tagasiteel pealinna Colombosse .,syntax,...,NaN,no,no,no,yes,no,yes,no,L,s_n80
